# Repeat a procedural Frozen Lake task

[repeat-task-gym](https://github.com/micahr234/repeat-task-gym) groups episodes into a task. Mouse-gym does not. This notebook builds a `RepeatTaskEnv` around [Procedural Frozen Lake](https://github.com/micahr234/procedural-frozenlake) and hands that Gymnasium env to `make_env` through `env_fn`.

`RepeatTaskEnv` puts episode end inside the observation (`episode_terminated`, `episode_truncated`) and uses Gymnasium `terminated` / `truncated` for the task. Mouse-gym forwards the observation, those flags, and `info`. `env.metrics` records a segment only when the Gymnasium flags are true — here, when the task ends. The next `step()` is a reset frame that starts another task (`env.step_index == 0`). There is no public `reset()`.

The lake keeps one map until `reset(options={"regenerate_map": True})`. Episode resets inside the task do not pass that option, so both episodes are the same lake. `EnvConfig.reset_options` is forwarded on every reset mouse-gym issues, so each new task gets a new map.


In [1]:
import gymnasium as gym
from procedural_frozenlake import ensure_registered

from mouse_gym import EnvConfig, make_env
from repeat_task_gym import RepeatTaskEnv

ensure_registered()


def make_task():
    lake = gym.make(
        "Procedural-FrozenLake-v1",
        map_seed=0,
        width=5,
        height=5,
        min_border=1,
        max_border=1,
        shoreline_jaggedness=0,
        max_episode_steps=8,
        emit_map=True,
    )
    return RepeatTaskEnv(
        lake,
        max_task_episodes=2,
        episode_seed_mode="per_task",
    )


env = make_env(
    EnvConfig(
        seed=0,
        name="repeat-frozen-lake",
        env_fn=make_task,
        reset_options={"regenerate_map": True},
    )
)


## Roll through two tasks

Each task is two episodes of one generated lake (`max_task_episodes=2`). `episode_seed_mode="per_task"` reuses one reset seed for both, so the start tile matches. `max_episode_steps=8` ends a wandering episode at the step limit.

Between episodes, `RepeatTaskEnv` returns a reset frame from `step()` (reward `0`, episode flags false, Gymnasium flags false). `env.step_index` keeps counting through that frame, because the wrapped env did not set `terminated` or `truncated`. The board in `info["map"]` stays the same.

The second episode sets `truncated=True`. That step appends to `env.metrics`. The length counts every step since mouse-gym's own reset, including the reset frame between the episodes. The following `step()` starts the next task at `step_index=0`, and `regenerate_map` draws a different board.

In this seeded run the first task times out both episodes, so the return is `0.0` and the length is `17` (eight steps, the reset frame, eight more steps). The second task times out, then reaches the goal (`reward=1.0`, `episode_terminated=True`); its length is `13`. The loop prints task resets, episode-end steps, and the reset frame between episodes.


In [2]:
env.action_space.seed(0)
seen_tasks = 0
show_reset_frame = False
try:
    for _ in range(200):
        observation, reward, terminated, truncated, info = env.step(env.action_space.sample())
        episode_terminated = bool(observation["episode_terminated"])
        episode_truncated = bool(observation["episode_truncated"])
        episode_end = episode_terminated or episode_truncated
        if env.step_index == 0 or episode_end or show_reset_frame:
            print(
                f"step={env.step_index} "
                f"reward={reward:.1f} "
                f"terminated={terminated} "
                f"truncated={truncated} "
                f"episode_terminated={episode_terminated} "
                f"episode_truncated={episode_truncated} "
                f"board={info['map']['board']} "
                f"returns={list(env.metrics.episode_cum_rewards)} "
                f"lengths={list(env.metrics.episode_lengths)}"
            )
        show_reset_frame = episode_end and not (terminated or truncated)
        if terminated or truncated:
            seen_tasks += 1
        if seen_tasks >= 2 and env.step_index == 0:
            break
finally:
    env.close()


step=0 reward=0.0 terminated=False truncated=False episode_terminated=False episode_truncated=False board=['TTTTT', 'TFFFT', 'TGFFT', 'TFFST', 'TTTTT'] returns=[] lengths=[]
step=8 reward=0.0 terminated=False truncated=False episode_terminated=False episode_truncated=True board=['TTTTT', 'TFFFT', 'TGFFT', 'TFFST', 'TTTTT'] returns=[] lengths=[]
step=9 reward=0.0 terminated=False truncated=False episode_terminated=False episode_truncated=False board=['TTTTT', 'TFFFT', 'TGFFT', 'TFFST', 'TTTTT'] returns=[] lengths=[]
step=17 reward=0.0 terminated=False truncated=True episode_terminated=False episode_truncated=True board=['TTTTT', 'TFFFT', 'TGFFT', 'TFFST', 'TTTTT'] returns=[0.0] lengths=[17.0]
step=0 reward=0.0 terminated=False truncated=False episode_terminated=False episode_truncated=False board=['TTTTT', 'THGFT', 'THFFT', 'THFST', 'TTTTT'] returns=[0.0] lengths=[17.0]
step=8 reward=0.0 terminated=False truncated=False episode_terminated=False episode_truncated=True board=['TTTTT', 'TH